In [1]:
import pandas as pd
import json

In [2]:
def read_jsonl_data(path : str, n :int = None) -> pd.DataFrame:
    data = []
    with open(path, 'r') as f:
        for i, line in enumerate(f):
            if n is not None and i >= n:
                break
            data.append(json.loads(line))

    return pd.DataFrame(data)

In [3]:
df = read_jsonl_data('/drive1/cuongtm/ntat/MulVulMoe/dataset/MulVulEx/train_022_full.jsonl')

In [4]:
df

,code,label,CWE_ID,lang
0,find_start_of_next_routerstatus(const char *s)...,0,unknown,c
1,IntSize DateTimeChooserImpl::contentSize()\n{\...,0,cwe-022,c
2,"_g_mime_type_get_from_content (char *buffer,\...",0,cwe-022,c
3,**/\n static void save_empty_cimg(std::...,0,cwe-787,c
4,FileEnumerator::~FileEnumerator() {\n}\n,0,cwe-022,c
...,...,...,...,...
3250,struct logicalVolIntegrityDescImpUse *udf_sb_l...,0,cwe-119,c
3251,static ssize_t iowarrior_write(struct file *fi...,0,cwe-416,c
3252,static ssize_t fuse_file_aio_write(struct kioc...,0,cwe-119,c
3253,template <> inline any &any_cast<any>(any &val...,0,cwe-476,c


In [5]:
print(df['lang'].value_counts())
print(df['label'].value_counts())
print(df['CWE_ID'].value_counts())

lang
c         2988
python     267
Name: count, dtype: int64
label
0    3199
1      56
Name: count, dtype: int64
CWE_ID
cwe-022    1030
unknown     238
cwe-119     208
cwe-20      161
cwe-125     153
           ... 
cwe-281       1
cwe-707       1
cwe-131       1
cwe-798       1
cwe-203       1
Name: count, Length: 88, dtype: int64


In [6]:
val_df = read_jsonl_data('/drive1/cuongtm/ntat/MulVulMoe/dataset/MulVulEx/val_022.jsonl')

In [8]:
print(val_df['lang'].value_counts())
print(val_df['label'].value_counts())
print(val_df['CWE_ID'].value_counts())

lang
python    63
Name: count, dtype: int64
label
0    55
1     8
Name: count, dtype: int64
CWE_ID
cwe-089    32
cwe-078    13
cwe-022    11
cwe-079     7
Name: count, dtype: int64


In [3]:
test_df = read_jsonl_data('/drive1/cuongtm/ntat/MulVulMoe/dataset/sven0802/cwe-022_test.jsonl')

In [5]:
test_df.columns = ['code', 'label', 'CWE_ID']
test_df['lang'] = 'python'

In [6]:
test_df

,code,label,CWE_ID,lang
0,@staticmethod\n def estimate_size(task_...,0,cwe-022,python
1,async def save(request):\n # TODO csrf\n ...,0,cwe-022,python
2,def pascal_case(value: str) -> str:\n retur...,0,cwe-022,python
3,"def set(self, key, value, replace=False):\...",1,cwe-022,python
4,"def list(self, keyfilter='/'):\n pa...",0,cwe-022,python
5,def create_basename_core(basename):\n try:\...,1,cwe-022,python
6,"def get(self, path):\n path = self....",0,cwe-022,python
7,"def candidate_paths_for_url(self, url):\n ...",0,cwe-022,python
8,"def _inject_net_into_fs(net, fs, execute=None)...",0,cwe-022,python
9,"def _inject_file_into_fs(fs, path, contents):\...",1,cwe-022,python


In [7]:
test_df.to_json('/drive1/cuongtm/ntat/MulVulMoe/dataset/MulVulEx/test_022.jsonl', orient='records', lines=True, force_ascii=False)

In [2]:
import os
import json

def normalize_cwe(cwe_val):
    if cwe_val is None:
        return None
    # Nếu là list thì lấy phần tử đầu tiên
    if isinstance(cwe_val, list):
        if len(cwe_val) == 0:
            return None
        cwe_val = cwe_val[0]
    # Nếu là số thì chuyển sang string
    if not isinstance(cwe_val, str):
        cwe_val = str(cwe_val)
    return cwe_val.strip("[]").lower()


def process_file(input_path: str, output_path: str):
    with open(input_path, 'r') as fin, open(output_path, 'w') as fout:
        for line in fin:
            obj = json.loads(line)
            new_obj = {
                "code": obj.get("func"),
                "label": obj.get("target"),
                "CWE_ID": normalize_cwe(obj.get("cwe"))
            }
            fout.write(json.dumps(new_obj) + "\n")

def map_output_name(filename: str) -> str:
    # filename ví dụ: primevul_train.jsonl, primevul_valid_paired.jsonl
    name = filename.replace("primevul_", "")
    # đổi valid -> val
    name = name.replace("valid", "val")
    # đổi phần mở rộng
    name = name.replace(".jsonl", "_norm.jsonl")
    return name

folder = "/drive1/cuongtm/ntat/Archive/PrimeVulRaw"
for file in os.listdir(folder):
    if file.endswith(".jsonl"):
        input_path = os.path.join(folder, file)
        output_name = map_output_name(file)
        output_path = os.path.join(folder, output_name)
        process_file(input_path, output_path)
        print(f"Processed {file} -> {output_name}")


Processed train_paired_norm.jsonl -> train_paired_norm_norm.jsonl
Processed primevul_train_paired.jsonl -> train_paired_norm.jsonl
Processed primevul_test_paired.jsonl -> test_paired_norm.jsonl
Processed primevul_valid.jsonl -> val_norm.jsonl
Processed primevul_test.jsonl -> test_norm.jsonl
Processed primevul_valid_paired.jsonl -> val_paired_norm.jsonl
Processed primevul_train.jsonl -> train_norm.jsonl


In [1]:
import os
import json
import pandas as pd

def read_jsonl_data(path: str, n: int = None) -> pd.DataFrame:
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if n is not None and i >= n:
                break
            data.append(json.loads(line))
    return pd.DataFrame(data)


def process_folder(folder_path: str, n: int = None):
    for file_name in os.listdir(folder_path):
        if not file_name.endswith(".jsonl"):
            continue

        file_path = os.path.join(folder_path, file_name)
        print(f"Processing: {file_path}")

        # đọc streaming
        df = read_jsonl_data(file_path, n)

        # rename cột (đảm bảo đủ 3 cột)
        df.columns = ['code', 'label', 'CWE_ID']

        # lưu lại đè lên file cũ (jsonl)
        with open(file_path, 'w', encoding='utf-8') as f:
            for _, row in df.iterrows():
                json_line = {
                    "code": row['code'],
                    "label": row['label'],
                    "CWE_ID": row['CWE_ID']
                }
                f.write(json.dumps(json_line, ensure_ascii=False) + "\n")


# chạy
folder_path = "/drive1/cuongtm/ntat/MulVulMoe/dataset/sven0802/"
process_folder(folder_path)

Processing: /drive1/cuongtm/ntat/MulVulMoe/dataset/sven0802/cwe-022_test.jsonl
Processing: /drive1/cuongtm/ntat/MulVulMoe/dataset/sven0802/cwe-079_test.jsonl
Processing: /drive1/cuongtm/ntat/MulVulMoe/dataset/sven0802/cwe-078_test.jsonl
Processing: /drive1/cuongtm/ntat/MulVulMoe/dataset/sven0802/cwe-089_test.jsonl
